# 🚶‍♂️ Day 04: Queue Fundamentals — First In, First Out

---

## Stacks vs Queues — The Real Difference

Stack = LIFO (plate stack). Queue = FIFO (waiting line at a counter).

```
Stack:    add/remove from same end       [→ top →]
Queue:    add at back, remove from front  [front → ... → back]
```

In Python, use `collections.deque` for a proper queue — O(1) at both ends. **Never use `list.pop(0)`** — that's O(n) because it shifts everything.

```python
from collections import deque
q = deque()
q.append(1)      # Enqueue at back
q.popleft()       # Dequeue from front — O(1)!
```

**Where queues show up:** BFS traversal (the reason trees/graphs use queues), scheduling problems, and whenever you need to process things in arrival order.

---

## 🎯 Problem 1: Implement Queue Using Stacks (LC #232)

Can you build a FIFO queue using only LIFO stacks? Yes — with two stacks.

**The idea:** One stack for "input" (where we push), one for "output" (where we pop). When output is empty, pour all of input into output. This reverses the order — giving us FIFO!

```
Push 1, 2, 3:
  input_stack = [1, 2, 3],  output_stack = []

Pop (output empty, so pour):
  input_stack = [],  output_stack = [3, 2, 1]
  Pop from output → returns 1 ✅ (FIFO!)
```

**Amortized O(1):** Each element is moved from input to output at most once.

In [ ]:
class MyQueue:
    """
    Queue using two stacks. Amortized O(1) per operation.
    """
    def __init__(self):
        self.input_stack = []   # Push goes here
        self.output_stack = []  # Pop/Peek comes from here
    
    def push(self, x):
        self.input_stack.append(x)
    
    def _transfer(self):
        """Move input → output only when output is empty."""
        if not self.output_stack:
            while self.input_stack:
                self.output_stack.append(self.input_stack.pop())
    
    def pop(self):
        self._transfer()
        return self.output_stack.pop()
    
    def peek(self):
        self._transfer()
        return self.output_stack[-1]
    
    def empty(self):
        return not self.input_stack and not self.output_stack

# Test it
q = MyQueue()
q.push(1)
q.push(2)
print(q.peek())   # 1
print(q.pop())    # 1
print(q.empty())  # False
print(q.pop())    # 2
print(q.empty())  # True

**Interview follow-up:** "Why not transfer on every pop?" Because that's O(n) per operation. The lazy transfer (only when output is empty) gives amortized O(1).

---

## 🎯 Problem 2: Implement Stack Using Queues (LC #225)

The reverse — build LIFO using FIFO. The elegant approach uses a single queue:

**After each push, rotate the queue** so the new element ends up at the front. Then pop is just a regular dequeue.

In [ ]:
from collections import deque

class MyStack:
    """
    Stack using a single queue.
    Push is O(n), Pop is O(1).
    """
    def __init__(self):
        self.q = deque()
    
    def push(self, x):
        self.q.append(x)
        # Rotate: move all old elements behind the new one
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
    
    def pop(self):
        return self.q.popleft()
    
    def top(self):
        return self.q[0]
    
    def empty(self):
        return len(self.q) == 0

s = MyStack()
s.push(1)
s.push(2)
print(s.top())    # 2
print(s.pop())    # 2
print(s.empty())  # False

---

## 🎯 Problem 3: Design Circular Queue (LC #622)

A circular queue (ring buffer) wraps around when it reaches the end of the array. Used everywhere in OS buffers, network queues, etc.

**Key insight:** Use modulo arithmetic (`% capacity`) to wrap indices.

```
Capacity = 4
       ┌───┬───┬───┬───┐
       │ 0 │ 1 │ 2 │ 3 │
       └───┴───┴───┴───┘
 front ↑       ↑ rear
 
 After wrap: index 4 → 4 % 4 = 0 (back to start!)
```

In [ ]:
class MyCircularQueue:
    def __init__(self, k):
        self.data = [0] * k
        self.capacity = k
        self.front = 0
        self.size = 0
    
    def enQueue(self, value):
        if self.isFull():
            return False
        # Rear position = (front + size) % capacity
        rear = (self.front + self.size) % self.capacity
        self.data[rear] = value
        self.size += 1
        return True
    
    def deQueue(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.capacity
        self.size -= 1
        return True
    
    def Front(self):
        return -1 if self.isEmpty() else self.data[self.front]
    
    def Rear(self):
        if self.isEmpty():
            return -1
        rear = (self.front + self.size - 1) % self.capacity
        return self.data[rear]
    
    def isEmpty(self):
        return self.size == 0
    
    def isFull(self):
        return self.size == self.capacity

# Test it
cq = MyCircularQueue(3)
print(cq.enQueue(1))  # True
print(cq.enQueue(2))  # True
print(cq.enQueue(3))  # True
print(cq.enQueue(4))  # False (full!)
print(cq.Rear())      # 3
print(cq.isFull())    # True
print(cq.deQueue())   # True
print(cq.enQueue(4))  # True (wraps around!)
print(cq.Rear())      # 4

---

## 🗺️ Queue Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  QUEUE PATTERNS                                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. BFS TRAVERSAL: Queue = level-by-level exploration               ║
║     Trees (level order), Graphs (shortest path in unweighted)       ║
║                                                                      ║
║  2. QUEUE ↔ STACK CONVERSION: Two stacks = queue, rotate = stack   ║
║     Classic design question, tests understanding of FIFO/LIFO       ║
║                                                                      ║
║  3. CIRCULAR QUEUE: Modulo arithmetic for wrap-around               ║
║     rear = (front + size) % capacity                                ║
║                                                                      ║
║  4. PYTHON: Use deque, never list.pop(0)                            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 232 | Implement Queue using Stacks | Easy | Two stacks, lazy transfer |
| 225 | Implement Stack using Queues | Easy | Single queue rotation |
| 622 | Design Circular Queue | Medium | Ring buffer, modulo |
| 641 | Design Circular Deque | Medium | Circular + both ends |
| 933 | Number of Recent Calls | Easy | Queue with time window |
| 346 | Moving Average from Data Stream | Easy | Sliding window queue |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Queue = FIFO. In Python: collections.deque, not list.pop(0)   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Two stacks → Queue: lazy transfer for amortized O(1)          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Circular queue: modulo wraps index around the array            │
├──────────────────────────────────────────────────────────────────────┤
│  📌 BFS = Queue. DFS = Stack/Recursion. That's the core link.     │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Stack/Queue Patterns

Sliding window maximum with monotonic deque, and more design challenges.

---

### 🎉 Queues — Simple but Fundamental!
BFS, scheduling, buffers — FIFO is everywhere.